In [1]:
#Import cobra and load existing model will add on to
import cobra
from cobra import Gene, Reaction, Model, Metabolite 
model = cobra.io.read_sbml_model('i_ZM478_fixed.xml')


Set parameter Username
Set parameter LicenseID to value 2732945
Academic license - for non-commercial use only - expires 2026-11-04


In [2]:
#Checking contents of model
model

#Alternative way to just look at number genes/reactions/metabolites
# print(len(model.genes))
# print(len(model.reactions))
# print(len(model.metabolites))

Name,iZM4_478
Memory address,186c759d0
Number of metabolites,799
Number of reactions,857
Number of genes,479
Number of groups,39
Objective expression,1.0*BIOMASS_ZM - 1.0*BIOMASS_ZM_reverse_34f17
Compartments,"Extracellular, Cytoplasm, Periplasm"


In [3]:
#In the orginal GSM, several spontaneous or transport reactions have a placeholder GPR of s0001
#Going to remove GPR for these reactions and let them be orphan reactions
#Doing this will allow for the model to be read (inlcuding GPRs) and used as framework for RBA

# Check number of Reactions for s0001 (Spontenous Genes)
model.genes.s0001

# Change GRP into ' ', empty string for Spontenous Reactions with s0001 (20 Reactions).
model.reactions.ETOHtrpp.gene_reaction_rule = ''
model.reactions.G5SADs.gene_reaction_rule = ''
model.reactions.MEOHtrpp.gene_reaction_rule = ''
model.reactions.H2tpp.gene_reaction_rule = ''
model.reactions.GLYCtpp.gene_reaction_rule = ''
model.reactions.METOX2s.gene_reaction_rule = ''
model.reactions.NOtpp.gene_reaction_rule = ''
model.reactions.H2St1pp.gene_reaction_rule = ''
model.reactions.H2Otpp.gene_reaction_rule = ''
model.reactions.GTPHs.gene_reaction_rule = ''
model.reactions.NH4tpp.gene_reaction_rule = ''
model.reactions.CO2tpp.gene_reaction_rule = ''
model.reactions.ACALDtpp.gene_reaction_rule = ''
model.reactions.N2Otpp.gene_reaction_rule = ''
model.reactions.O2tpp.gene_reaction_rule = ''
model.reactions.H2Otex.gene_reaction_rule = ''
model.reactions.OMCDC.gene_reaction_rule = ''
model.reactions.FALDtpp.gene_reaction_rule = ''
model.reactions.METOX1s.gene_reaction_rule = ''
model.reactions.SO2tpp.gene_reaction_rule = ''

# Re-Check number of reactons with GPR, s0001.
model.genes.s0001

Gene identifier,s0001
Name,s0001
Memory address,0x186daf390
Functional,True
In 0 reaction(s),


In [4]:
#Now going to remove s0001 gene from the model so doesn't accidently get read into RBA
cobra.manipulation.remove_genes(model, ['s0001'])

#Check that is was removed: if uncomment line below should get error that couldn't find s0001
#model.genes.s0001

In [5]:
#For the pantothenate synthase (PANTS) reaction will replace the GPR of ZMO1791 with ZMO1954
#Doing this for the RBA model because in Uniprot ZMO1971 has no reported protein while ZMO1954 does
#ZMO1954 annotated as panC gene (encoding pantoate--beta-alanine ligase) while ZMO1971 annotated as a putative transporter with no associated protein in Uniprot database

print(model.reactions.PANTS.gene_reaction_rule)

model.reactions.PANTS.gene_reaction_rule = 'ZMO1954'

print(model.reactions.PANTS.gene_reaction_rule)

#Removing ZMO1971 from GSM so not accidently read into RBA model build, was GPR for PANTS but replaced with ZMO1954 (1971 has no reported protein in uniprot while 1954 does)
cobra.manipulation.remove_genes(model, ['ZMO1971'])

ZMO1971
ZMO1954


In [6]:
#Creating two separate ispG reactions: one using fldA and one using fdxA
#unlinking GPR for fldA or fdxA
ispG_fld = model.reactions.get_by_id('MECDPDH5')
ispG_fdx = ispG_fld.copy()

ispG_fdx.id = 'MECDPDH5_fdx'
ispG_fdx.name = '(E)-4-hydroxy-3-methylbut-2-en-1-yl-diphosphate synthase (ferredoxin)'

ispG_fld.name = '(E)-4-hydroxy-3-methylbut-2-en-1-yl-diphosphate synthase (flavodoxin)'
ispG_fld.id = 'MECDPDH5_fld'

#print(ispG_fdx)
#print(ispG_fld)

ispG_fld.gene_reaction_rule = ('ZMO0180')
ispG_fdx.gene_reaction_rule = ('ZMO0180')

ispG_fdx.subtract_metabolites({model.metabolites.get_by_id('flxr_c'): -2})
ispG_fdx.subtract_metabolites({model.metabolites.get_by_id('flxso_c'): 2})
ispG_fdx.add_metabolites({model.metabolites.get_by_id('fdxr__4_2_c'): -1})
ispG_fdx.add_metabolites({model.metabolites.get_by_id('fdxo__4_2_c'): 1})

#print(ispG_fdx.check_mass_balance())
#print(ispG_fld.check_mass_balance())
#print(ispG_fdx)
#print(ispG_fld)
#print(ispG_fdx.gene_reaction_rule)
#print(ispG_fld.gene_reaction_rule)

In [7]:
#Creating two separate ispH reactions: one using fldA and one using fdxA
#Replacing old ispH reaction that using nadh
#unlinking GPR for fldA or fdxA
ispH_IPP_fld = model.reactions.get_by_id('IPDPS')
ispH_IPP_fdx = ispH_IPP_fld.copy()

ispH_DPP_fld = model.reactions.get_by_id('DMPPS')
ispH_DPP_fdx = ispH_DPP_fld.copy()

ispH_IPP_fdx.id = 'IPDPS_fdx'
ispH_IPP_fdx.name = '1-hydroxy-2-methyl-2-(E)-butenyl 4-diphosphate reductase (ipdp, ferredoxin)'

ispH_IPP_fld.name = '1-hydroxy-2-methyl-2-(E)-butenyl 4-diphosphate reductase (ipdp, flavodoxin)'
ispH_IPP_fld.id = 'IPDPS_fld'

ispH_DPP_fdx.id = 'DMPPS_fdx'
ispH_DPP_fdx.name = '1-hydroxy-2-methyl-2-(E)-butenyl 4-diphosphate reductase (dmpps, ferredoxin)'

ispH_DPP_fld.name = '1-hydroxy-2-methyl-2-(E)-butenyl 4-diphosphate reductase (dmpps, flavodoxin)'
ispH_DPP_fld.id = 'DMPPS_fld'

ispH_IPP_fld.subtract_metabolites({model.metabolites.get_by_id('nadh_c'): -1})
ispH_IPP_fld.subtract_metabolites({model.metabolites.get_by_id('nad_c'): 1})
ispH_IPP_fld.add_metabolites({model.metabolites.get_by_id('flxr_c'): -2})
ispH_IPP_fld.add_metabolites({model.metabolites.get_by_id('flxso_c'): 2})
ispH_IPP_fld.add_metabolites({model.metabolites.get_by_id('h_c'): -1})

ispH_IPP_fdx.subtract_metabolites({model.metabolites.get_by_id('nadh_c'): -1})
ispH_IPP_fdx.subtract_metabolites({model.metabolites.get_by_id('nad_c'): 1})
ispH_IPP_fdx.add_metabolites({model.metabolites.get_by_id('fdxr__4_2_c'): -1})
ispH_IPP_fdx.add_metabolites({model.metabolites.get_by_id('fdxo__4_2_c'): 1})
ispH_IPP_fdx.add_metabolites({model.metabolites.get_by_id('h_c'): -1})

ispH_DPP_fld.subtract_metabolites({model.metabolites.get_by_id('nadh_c'): -1})
ispH_DPP_fld.subtract_metabolites({model.metabolites.get_by_id('nad_c'): 1})
ispH_DPP_fld.add_metabolites({model.metabolites.get_by_id('flxr_c'): -2})
ispH_DPP_fld.add_metabolites({model.metabolites.get_by_id('flxso_c'): 2})
ispH_DPP_fld.add_metabolites({model.metabolites.get_by_id('h_c'): -1})

ispH_DPP_fdx.subtract_metabolites({model.metabolites.get_by_id('nadh_c'): -1})
ispH_DPP_fdx.subtract_metabolites({model.metabolites.get_by_id('nad_c'): 1})
ispH_DPP_fdx.add_metabolites({model.metabolites.get_by_id('fdxr__4_2_c'): -1})
ispH_DPP_fdx.add_metabolites({model.metabolites.get_by_id('fdxo__4_2_c'): 1})
ispH_DPP_fdx.add_metabolites({model.metabolites.get_by_id('h_c'): -1})

# print(ispH_IPP_fld.check_mass_balance())
# print(ispH_IPP_fdx.check_mass_balance())
# print(ispH_IPP_fld.reaction)
# print(ispH_IPP_fdx.reaction)
# print(ispH_IPP_fld.gene_reaction_rule)
# print(ispH_IPP_fdx.gene_reaction_rule)

# print(ispH_DPP_fld.check_mass_balance())
# print(ispH_DPP_fdx.check_mass_balance())
# print(ispH_DPP_fld.reaction)
# print(ispH_DPP_fdx.reaction)
#print(ispH_DPP_fld.gene_reaction_rule)
# print(ispH_DPP_fdx.gene_reaction_rule)

In [8]:
#Creating two separate HPN1 reactions: one using fldA and one using fdxA
hpn_fld = model.reactions.get_by_id('HPN1')
hpn_fdx = hpn_fld.copy()

hpn_fdx.id = 'HPN1_fdx'
hpn_fdx.name = 'Adenosylhopane synthase (ferredoxin)'

hpn_fld.name = 'Adenosylhopane synthase (flavodoxin)'
hpn_fld.id = 'HPN1_fld'

#print(hpn_fdx)
#print(hpn_fld)


hpn_fdx.subtract_metabolites({model.metabolites.get_by_id('flxr_c'): -2})
hpn_fdx.subtract_metabolites({model.metabolites.get_by_id('flxso_c'): 2})
hpn_fdx.add_metabolites({model.metabolites.get_by_id('fdxr__4_2_c'): -1})
hpn_fdx.add_metabolites({model.metabolites.get_by_id('fdxo__4_2_c'): 1})

# print(hpn_fdx.check_mass_balance())
# print(hpn_fld.check_mass_balance())
# print(hpn_fdx)
# print(hpn_fld)


In [9]:
#Creating two separate nitrogenase reactions (fld or fdx)
nit_fdx = model.reactions.get_by_id('NIT1b')
nit_fld = nit_fdx.copy()

nit_fdx.id = 'NIT1b_fdx'
nit_fdx.name = 'nitrogenase (ferredoxin)'

nit_fld.name = 'nitrogenase (flavodoxin)'
nit_fld.id = 'NIT1b_fld'

print(nit_fdx)
print(nit_fld)


nit_fld.subtract_metabolites({model.metabolites.get_by_id('fdxr__4_2_c'): -4})
nit_fld.subtract_metabolites({model.metabolites.get_by_id('fdxo__4_2_c'): 4})
nit_fld.add_metabolites({model.metabolites.get_by_id('flxr_c'): -8})
nit_fld.add_metabolites({model.metabolites.get_by_id('flxso_c'): 8})

print(nit_fdx.check_mass_balance())
print(nit_fld.check_mass_balance())
print(nit_fdx)
print(nit_fld)

NIT1b_fdx: 16.0 atp_c + 4.0 fdxr__4_2_c + 16.0 h2o_c + n2_c <=> 16.0 adp_c + 4.0 fdxo__4_2_c + h2_c + 6.0 h_c + 2.0 nh4_c + 16.0 pi_c
NIT1b_fld: 16.0 atp_c + 4.0 fdxr__4_2_c + 16.0 h2o_c + n2_c <=> 16.0 adp_c + 4.0 fdxo__4_2_c + h2_c + 6.0 h_c + 2.0 nh4_c + 16.0 pi_c
{}
{}
NIT1b_fdx: 16.0 atp_c + 4.0 fdxr__4_2_c + 16.0 h2o_c + n2_c <=> 16.0 adp_c + 4.0 fdxo__4_2_c + h2_c + 6.0 h_c + 2.0 nh4_c + 16.0 pi_c
NIT1b_fld: 16.0 atp_c + 8 flxr_c + 16.0 h2o_c + n2_c <=> 16.0 adp_c + 8 flxso_c + h2_c + 6.0 h_c + 2.0 nh4_c + 16.0 pi_c


In [10]:
#Adding new reactions to the model
model.add_reactions([ispG_fdx, ispH_IPP_fdx, ispH_DPP_fdx, hpn_fdx, nit_fld])

In [11]:
#Adding fdxA (ZMO0220) gene to model: adding it as orphan gene (not assigned to reactions, no GPR) since will be using it as modeled protein in RBA
fdxA_gene = Gene('ZMO0220',name = '4Fe-4S ferredoxin iron-sulfur binding domain protein (fdxA)')
model.genes.add(fdxA_gene)
#print(fdxA_gene.name)
#print(fdxA_gene.reactions)

In [12]:
#Modelilng flavodoxin and ferredoxin oxidoreductase reactions 
#If allow them to go in either direction, the model chooses these reactions to synthesize reduced fld/fdx since it is more 'free' than using RNF so also turn off one direction when run
#for each GPR will be ZMO1753 (fld/fdx oxidoreductase)
FNOR = model.reactions.get_by_id('FNOR')
FNOR.gene_reaction_rule = ('ZMO1753')
FNOR.upper_bound = 1000
FNOR.lower_bound = -1000
print(FNOR)
#print(FNOR.gene_reaction_rule)

FLDR = model.reactions.get_by_id('FLDR2')
FLDR.gene_reaction_rule = ('ZMO1753')
FLDR.upper_bound = 1000
FLDR.lower_bound = -1000
print(FLDR)
#print(FLDR.gene_reaction_rule)

FNOR: fdxr__4_2_c + h_c + nadp_c <=> fdxo__4_2_c + nadph_c
FLDR2: 2.0 flxso_c + nadph_c <=> 2.0 flxr_c + h_c + nadp_c


In [13]:
#Removing fldA GPR and from RNTR reactions and creating a ferredoxin version
RNTR2_fld = model.reactions.get_by_id('RNTR2c2')
RNTR2_fld.gene_reaction_rule = ('ZMO1025')
#print(RNTR2_fld)
#print(RNTR2_fld.gene_reaction_rule)
RNTR2_fdx = RNTR2_fld.copy()

RNTR2_fdx.id = 'RNTR2c2_fdx'
RNTR2_fdx.name = 'ribonucleoside-triphopshate reductase (GTP) (ferredoxin)'

RNTR2_fld.id = 'RNTR2c2_fld'
RNTR2_fld.name = 'ribonucleoside-triphopshate reductase (GPT) (flavodoxin)'

print(RNTR2_fld)
print(RNTR2_fdx)


RNTR2_fdx.subtract_metabolites({model.metabolites.get_by_id('flxr_c'): -2})
RNTR2_fdx.subtract_metabolites({model.metabolites.get_by_id('flxso_c'): 2})
RNTR2_fdx.add_metabolites({model.metabolites.get_by_id('fdxr__4_2_c'): -1})
RNTR2_fdx.add_metabolites({model.metabolites.get_by_id('fdxo__4_2_c'): 1})

print(RNTR2_fdx.check_mass_balance())
print(RNTR2_fld.check_mass_balance())
print(RNTR2_fdx)
print(RNTR2_fld)

RNTR1_fld = model.reactions.get_by_id('RNTR1c2')
RNTR1_fld.gene_reaction_rule = ('ZMO1025')
#print(RNTR1_fld)
#print(RNTR1_fld.gene_reaction_rule)
RNTR1_fdx = RNTR1_fld.copy()

RNTR1_fdx.id = 'RNTR1c2_fdx'
RNTR1_fdx.name = 'ribonucleoside-triphopshate reductase (ATP) (ferredoxin)'

RNTR1_fld.id = 'RNTR1c2_fld'
RNTR1_fld.name = 'ribonucleoside-triphopshate reductase (ATP) (flavodoxin)'

print(RNTR1_fld)
print(RNTR1_fdx)

RNTR1_fdx.subtract_metabolites({model.metabolites.get_by_id('flxr_c'): -2})
RNTR1_fdx.subtract_metabolites({model.metabolites.get_by_id('flxso_c'): 2})
RNTR1_fdx.add_metabolites({model.metabolites.get_by_id('fdxr__4_2_c'): -1})
RNTR1_fdx.add_metabolites({model.metabolites.get_by_id('fdxo__4_2_c'): 1})

print(RNTR1_fdx.check_mass_balance())
print(RNTR1_fld.check_mass_balance())
print(RNTR1_fdx)
print(RNTR1_fld)


RNTR3_fld = model.reactions.get_by_id('RNTR3c2')
RNTR3_fld.gene_reaction_rule = ('ZMO1025')
#print(RNTR3_fld)
#print(RNTR3.gene_reaction_rule)

RNTR3_fdx = RNTR3_fld.copy()

RNTR3_fdx.id = 'RNTR3c2_fdx'
RNTR3_fdx.name = 'ribonucleoside-triphopshate reductase (CTP) (ferredoxin)'

RNTR3_fld.id = 'RNTR3c3_fld'
RNTR3_fld.name = 'ribonucleoside-triphopshate reductase (CTP) (flavodoxin)'

print(RNTR3_fld)
print(RNTR3_fdx)

RNTR3_fdx.subtract_metabolites({model.metabolites.get_by_id('flxr_c'): -2})
RNTR3_fdx.subtract_metabolites({model.metabolites.get_by_id('flxso_c'): 2})
RNTR3_fdx.add_metabolites({model.metabolites.get_by_id('fdxr__4_2_c'): -1})
RNTR3_fdx.add_metabolites({model.metabolites.get_by_id('fdxo__4_2_c'): 1})

print(RNTR3_fdx.check_mass_balance())
print(RNTR3_fld.check_mass_balance())
print(RNTR3_fdx)
print(RNTR3_fld)


RNTR4_fld = model.reactions.get_by_id('RNTR4c2')
RNTR4_fld.gene_reaction_rule = ('ZMO1025')
#print(RNTR4_fld)
#print(RNTR4_fld.gene_reaction_rule)

RNTR4_fdx = RNTR4_fld.copy()

RNTR4_fdx.id = 'RNTR4c2_fdx'
RNTR4_fdx.name = 'ribonucleoside-triphopshate reductase (UTP) (ferredoxin)'

RNTR4_fld.id = 'RNTR4c2_fld'
RNTR4_fld.name = 'ribonucleoside-triphopshate reductase (UTP) (flavodoxin)'

print(RNTR4_fld)
print(RNTR4_fdx)

RNTR4_fdx.subtract_metabolites({model.metabolites.get_by_id('flxr_c'): -2})
RNTR4_fdx.subtract_metabolites({model.metabolites.get_by_id('flxso_c'): 2})
RNTR4_fdx.add_metabolites({model.metabolites.get_by_id('fdxr__4_2_c'): -1})
RNTR4_fdx.add_metabolites({model.metabolites.get_by_id('fdxo__4_2_c'): 1})

print(RNTR4_fdx.check_mass_balance())
print(RNTR4_fld.check_mass_balance())
print(RNTR4_fdx)
print(RNTR4_fld)

RNTR2c2_fld: 2.0 flxr_c + gtp_c + 2.0 h_c --> dgtp_c + 2.0 flxso_c + h2o_c
RNTR2c2_fdx: 2.0 flxr_c + gtp_c + 2.0 h_c --> dgtp_c + 2.0 flxso_c + h2o_c
{}
{}
RNTR2c2_fdx: fdxr__4_2_c + gtp_c + 2.0 h_c --> dgtp_c + fdxo__4_2_c + h2o_c
RNTR2c2_fld: 2.0 flxr_c + gtp_c + 2.0 h_c --> dgtp_c + 2.0 flxso_c + h2o_c
RNTR1c2_fld: atp_c + 2.0 flxr_c + 2.0 h_c --> datp_c + 2.0 flxso_c + h2o_c
RNTR1c2_fdx: atp_c + 2.0 flxr_c + 2.0 h_c --> datp_c + 2.0 flxso_c + h2o_c
{}
{}
RNTR1c2_fdx: atp_c + fdxr__4_2_c + 2.0 h_c --> datp_c + fdxo__4_2_c + h2o_c
RNTR1c2_fld: atp_c + 2.0 flxr_c + 2.0 h_c --> datp_c + 2.0 flxso_c + h2o_c
RNTR3c3_fld: ctp_c + 2.0 flxr_c + 2.0 h_c --> dctp_c + 2.0 flxso_c + h2o_c
RNTR3c2_fdx: ctp_c + 2.0 flxr_c + 2.0 h_c --> dctp_c + 2.0 flxso_c + h2o_c
{}
{}
RNTR3c2_fdx: ctp_c + fdxr__4_2_c + 2.0 h_c --> dctp_c + fdxo__4_2_c + h2o_c
RNTR3c3_fld: ctp_c + 2.0 flxr_c + 2.0 h_c --> dctp_c + 2.0 flxso_c + h2o_c
RNTR4c2_fld: 2.0 flxr_c + 2.0 h_c + utp_c --> dutp_c + 2.0 flxso_c + h2o_c
RNTR

In [14]:
#Adding new reactions to the model
model.add_reactions([RNTR1_fdx, RNTR2_fdx, RNTR3_fdx, RNTR4_fdx])

In [15]:
#Adding RNF complex (ZMO) genes and reaction to model
#Making it use Na (model has laread a na h antiporter) because if use H then what happens is burn through energy with ATP synthase
#Since all of the modeling will be under anaerobic conditions, will set the RNF complex to go in biosynthetic direction (making reduced fdx/fld) when run model
#We have experimental data showing when KD rnfE see accumulation of proton gradient and increase in MEcDP accumulation (Amy E. paper and Julio data)
RNF_fld = Reaction(
    'RNFfld',
    name = 'RNF electron transport complex (RNFABCDGEH), flavodoxin', 
    subsystem = 'Energy metabolism', 
    lower_bound= -1000, 
    upper_bound= 1000,
)    

RNF_fld.notes['SUBSYSTEM'] = 'Energy metabolism'

RNF_fld.add_metabolites({model.metabolites.get_by_id('h_p'): -3})
RNF_fld.add_metabolites({model.metabolites.get_by_id('flxso_c'): -2})
RNF_fld.add_metabolites({model.metabolites.get_by_id('nadh_c'): -1})
RNF_fld.add_metabolites({model.metabolites.get_by_id('flxr_c'): 2})
RNF_fld.add_metabolites({model.metabolites.get_by_id('nad_c'): 1})
RNF_fld.add_metabolites({model.metabolites.get_by_id('h_c'): 4})

RNF_fld.gene_reaction_rule = ('ZMO1814 and ZMO1813 and ZMO1812 and ZMO1811 and ZMO1810 and ZMO1809 and ZMO1808')

RNF_fdx = Reaction(
    'RNFfdx',
    name = 'RNF electron transport complex (RNFABCDGEH), ferredoxin', 
    subsystem = 'Energy metabolism', 
    lower_bound= -1000, 
    upper_bound= 1000,
)    

RNF_fdx.notes['SUBSYSTEM'] = 'Energy metabolism'

RNF_fdx.add_metabolites({model.metabolites.get_by_id('h_p'): -3})
RNF_fdx.add_metabolites({model.metabolites.get_by_id('fdxo__4_2_c'): -1})
RNF_fdx.add_metabolites({model.metabolites.get_by_id('nadh_c'): -1})
RNF_fdx.add_metabolites({model.metabolites.get_by_id('fdxr__4_2_c'): 1})
RNF_fdx.add_metabolites({model.metabolites.get_by_id('nad_c'): 1})
RNF_fdx.add_metabolites({model.metabolites.get_by_id('h_c'): 4})

RNF_fdx.gene_reaction_rule = ('ZMO1814 and ZMO1813 and ZMO1812 and ZMO1811 and ZMO1810 and ZMO1809 and ZMO1808')

model.add_reactions([RNF_fld, RNF_fdx])

# print(f"Reaction ID: {RNF_fld.id}")
# print(f"Assigned Name: '{RNF_fld.name}'")
# print(f"Assigned Subsystem: '{RNF_fld.subsystem}'")
# print(f"Assigned Bounds: {RNF_fld.bounds}")
# print("Reaction:", RNF_fld)
print(f"Mass Balance: {RNF_fld.check_mass_balance()}")
print(f"Mass Balance: {RNF_fdx.check_mass_balance()}")
# print(f"GPR: {RNF_fld.gene_reaction_rule}")

Mass Balance: {}
Mass Balance: {}


In [16]:
#Adding Asparaginyl-tRNA synthetase (asparagine synthetase/trna charing), id: ASNTRS
#This is needed for the RBA model since it explicitly models translation and the current GSM was lacking the asparaginyl-tRNA and glutaminyl-tRNA synthetases

asn__L_c = model.metabolites.get_by_id ('asn__L_c')
atp_c = model.metabolites.get_by_id ('atp_c')
amp_c = model.metabolites.get_by_id ('amp_c')
ppi_c = model.metabolites.get_by_id('ppi_c')
# Add new metabolites, trnasn_c and asntrna_c
trnaasn_c = Metabolite('trnaasn_c', formula='R', name='tRNA(Asn)', compartment ='c', charge=0)
asntrna_c = Metabolite('asntrna_c', formula='C4H7N2O2R', name='L-Asparaginyl-tRNA(Asn)', compartment ='c', charge=1)
#Define id, name, subsystem, boundary conditions
ASNTRS = Reaction(
    'ASNTRS',
    name = 'Asparaginyl-tRNA synthetase', 
    subsystem = 'tRNA Charging', 
    lower_bound= 0, 
    upper_bound= 1000,
)    

ASNTRS.notes['SUBSYSTEM'] = 'tRNA Charging'

#Add metabolites into the reaction.
ASNTRS.add_metabolites ({asn__L_c: -1.0, atp_c: -1.0,trnaasn_c:-1.0, amp_c:1.0, asntrna_c:1.0, ppi_c: 1.0 })
#Add reaction into the model
model.add_reactions([ASNTRS])
#Gene Information from Uniprot
ASNTRS.gene_reaction_rule = 'ZMO0782 or ZMO0784'

# print(ASNTRS.check_mass_balance())
# print(ASNTRS.gene_reaction_rule)


#Adding Glutaminyl-tRNA synthetase (Glutamine synthetase), id: GLNTRS
gln__L_c = model.metabolites.get_by_id ('gln__L_c')
atp_c = model.metabolites.get_by_id ('atp_c')
amp_c = model.metabolites.get_by_id ('amp_c')
ppi_c = model.metabolites.get_by_id('ppi_c')

# Add new metabolites, trnasn_c and asntrna_c
trnagln_c = Metabolite('trnagln_c', formula='R', name='tRNA(Gln)', compartment ='c', charge=0)
glntrna_c = Metabolite('glntrna_c', formula='C5H9N2O2R', name='L-Glutaminyl-tRNA(Gln)', compartment ='c', charge=1)

#Define id, name, subsystem, boundary conditions
GLNTRS = Reaction(
    'GLNTRS',
    name = 'Glutaminyl-tRNA synthetase', 
    subsystem = 'tRNA Charging', 
    lower_bound= 0, 
    upper_bound= 1000,
)    

GLNTRS.notes['SUBSYSTEM'] = 'tRNA Charging'

GLNTRS.add_metabolites ({gln__L_c: -1.0, atp_c: -1.0,trnagln_c:-1.0, amp_c:1.0, glntrna_c:1.0, ppi_c: 1.0 })
model.add_reactions([GLNTRS])
#Gene Information from Uniprot
GLNTRS.gene_reaction_rule = 'ZMO0782 or ZMO0783 or ZMO0784'

# print(GLNTRS.check_mass_balance())
# print(GLNTRS.gene_reaction_rule)



In [17]:
#For the reactions that synthesizes the C190cACP_c (C190cSN) need to add GPR (ZMO1033) since this is cfa gene (Cyclopropane-fatty-acyl-phospholipid synthase)
cfa = model.reactions.get_by_id('C190cSN')
cfa.gene_reaction_rule = ('ZMO1033')
#print(cfa.gene_reaction_rule)

In [18]:
model 

Name,iZM4_478
Memory address,186c759d0
Number of metabolites,803
Number of reactions,870
Number of genes,489
Number of groups,39
Objective expression,1.0*BIOMASS_ZM - 1.0*BIOMASS_ZM_reverse_34f17
Compartments,"Extracellular, Cytoplasm, Periplasm"


In [19]:
#Removing the OAADC reaction (oxaloacetate decarboxylase): there is no annotated gene for this reaction in KEGG ALSO the GPR they previously had in the model was for ZMO0997 which is EDA!!!
oaddc = model.reactions.get_by_id('OAADC')
print(oaddc)
oaddc.remove_from_model()



OAADC: h_c + oaa_c --> co2_c + pyr_c


/opt/anaconda3/envs/cobra_env/lib/python3.11/site-packages/cobra/core/group.py:147: UserWarning: need to pass in a list
  warn("need to pass in a list")


In [20]:
#Removing ZMO1722 from ALCD2x reactions (alcohol dehydrogense for ethanol)
#ZMO1722 is s-(hydroxymethyl)glutathione dehydrogenase which is used to detoxify formate and is already written as a reaction in model
#Assocaited with ALCD2x will only be the AdHA/B (I/II) which have experimental proof
adh = model.reactions.get_by_id('ALCD2x')
print(adh.gene_reaction_rule)
adh.gene_reaction_rule = ('ZMO1236 or ZMO1596')
print(adh.gene_reaction_rule)




ZMO1236 or ZMO1722 or ZMO1596
ZMO1236 or ZMO1596


In [21]:
#Removing ZMO0353 from MEPCT (IspF) reaction as it is misannotated as IspF in uniprot 
#Genebank and KEGG have it annotated as ribitol-5-PO4 dehydrogenase (NADP+) and D-ribito-5-PO4 cytidyltransferase
#so will add in these reactions: CDP-ribitol used in crating exopolyssacharides and cell wall structures so subsytem will be cell envelope biosynthesis (annotated in Kegg as part of capsule biosynthesis)
ispf = model.reactions.get_by_id('MEPCT')
print(ispf.gene_reaction_rule)
ispf.gene_reaction_rule = ('ZMO1128')
print(ispf.gene_reaction_rule)


rib5po4 = model.metabolites.get_by_id ('ru5p__D_c')
ctp = model.metabolites.get_by_id('ctp_c')
h = model.metabolites.get_by_id('h_c')
ppi = model.metabolites.get_by_id('ppi_c')
nadph = model.metabolites.get_by_id('nadph_c')
nadp = model.metabolites.get_by_id('nadp_c')


# Add new metabolites, trnasn_c and asntrna_c
ribitol5p_c = Metabolite('rib5p__D_c', formula='C5H11O8P', name='D-ribitol-5-phosphate', compartment ='c', charge=-2)
cdp_ribitol_c = Metabolite('CDP-rib_c', formula='C14H23N3O15P2', name='CDP-ribitol', compartment ='c', charge=-2)


#Define id, name, subsystem, boundary conditions
RIBDH = Reaction(
    'RIBDH',
    name = 'ribitol-5-PO4 2-dehydrogenase (NADP+)', 
    subsystem = 'cell envelope biosynthesis', 
    lower_bound= -1000, 
    upper_bound= 1000,
)    

RIBDH.notes['SUBSYSTEM'] = 'cell envelope biosynthesis'

#Add metabolites into the reaction.
RIBDH.add_metabolites ({rib5po4: -1.0, nadph:-1.0, h:-1.0, ribitol5p_c:1.0, nadp:1.0 })
RIBDH.gene_reaction_rule = ('ZMO0353')
#Add reaction into the model
model.add_reactions([RIBDH])
print(RIBDH)
print(RIBDH.check_mass_balance())
print(RIBDH.gene_reaction_rule)


#Define id, name, subsystem, boundary conditions
RIBCT = Reaction(
    'RIBCT',
    name = 'D-ribito-5-PO4 cytidyltransferase', 
    subsystem = 'cell envelope biosynthesis', 
    lower_bound= -1000, 
    upper_bound= 1000,
)    

RIBCT.notes['SUBSYSTEM'] = 'cell envelope biosynthesis'

#Add metabolites into the reaction.
RIBCT.add_metabolites ({ribitol5p_c: -1.0, ctp: -1.0,h:-1.0, cdp_ribitol_c:1.0, ppi:1.0 })
RIBCT.gene_reaction_rule = ('ZMO0353')
#Add reaction into the model
model.add_reactions([RIBCT])
print(RIBCT)
print(RIBCT.check_mass_balance())
print(RIBCT.gene_reaction_rule)




ZMO1128 or ZMO0353
ZMO1128
RIBDH: h_c + nadph_c + ru5p__D_c <=> nadp_c + rib5p__D_c
{}
ZMO0353
RIBCT: ctp_c + h_c + rib5p__D_c <=> CDP-rib_c + ppi_c
{}
ZMO0353


In [22]:
model

Name,iZM4_478
Memory address,186c759d0
Number of metabolites,805
Number of reactions,871
Number of genes,489
Number of groups,39
Objective expression,1.0*BIOMASS_ZM - 1.0*BIOMASS_ZM_reverse_34f17
Compartments,"Extracellular, Cytoplasm, Periplasm"


In [ ]:
#Saving updated model and run on Parameterizing_GAM_nGAM.py
#cobra.io.write_sbml_model(model, 'i_ZM4_489_debug.xml')

#So far it is running and solvable for low glucose uptake values so is indeed biomass issue
#Below slowly going to start changing biomass (begin with proteins and then will change lipids)

In [23]:
#Removing the protein biomass precursor reaction metabolites and rewriting to reflect the process of protein synthesis
#Also updated the coefficients (in mmol/g) to reflect our experimental quantitative proteomics data (although keeping the assumption that proteins make up 52.7% total biomass since our data give a value in 30s and it way too low for model to run)
#More details refer to 'KF_biomass updates_GSM_RBA.xlsx'
protein = model.reactions.get_by_id('PROTEINr_ZM')
protein.subtract_metabolites({model.metabolites.get_by_id('ala__L_c'): -0.954})
protein.subtract_metabolites({model.metabolites.get_by_id('arg__L_c'): -0.475})
protein.subtract_metabolites({model.metabolites.get_by_id('asn__L_c'): -0.319})
protein.subtract_metabolites({model.metabolites.get_by_id('asp__L_c'): -0.487})
protein.subtract_metabolites({model.metabolites.get_by_id('atp_c'): -7.997})
protein.subtract_metabolites({model.metabolites.get_by_id('cys__L_c'): -0.0708})
protein.subtract_metabolites({model.metabolites.get_by_id('gln__L_c'): -0.327})
protein.subtract_metabolites({model.metabolites.get_by_id('glu__L_c'): -0.486})
protein.subtract_metabolites({model.metabolites.get_by_id('gly_c'): -0.745})
protein.subtract_metabolites({model.metabolites.get_by_id('gtp_c'): -15.997})
protein.subtract_metabolites({model.metabolites.get_by_id('h2o_c'): -15.994})
protein.subtract_metabolites({model.metabolites.get_by_id('his__L_c'): -0.18})
protein.subtract_metabolites({model.metabolites.get_by_id('ile__L_c'): -0.478})
protein.subtract_metabolites({model.metabolites.get_by_id('leu__L_c'): -0.511})
protein.subtract_metabolites({model.metabolites.get_by_id('lys__L_c'): -0.51})
protein.subtract_metabolites({model.metabolites.get_by_id('met__L_c'): -0.181})
protein.subtract_metabolites({model.metabolites.get_by_id('phe__L_c'): -0.279})
protein.subtract_metabolites({model.metabolites.get_by_id('pro__L_c'): -0.39})
protein.subtract_metabolites({model.metabolites.get_by_id('ser__L_c'): -0.446})
protein.subtract_metabolites({model.metabolites.get_by_id('thr__L_c'): -0.436})
protein.subtract_metabolites({model.metabolites.get_by_id('trp__L_c'): -0.107})
protein.subtract_metabolites({model.metabolites.get_by_id('tyr__L_c'): -0.191})
protein.subtract_metabolites({model.metabolites.get_by_id('val__L_c'): -0.934})
protein.subtract_metabolites({model.metabolites.get_by_id('amp_c'): 7.997})
protein.subtract_metabolites({model.metabolites.get_by_id('gdp_c'): 15.994})
protein.subtract_metabolites({model.metabolites.get_by_id('h_c'): 15.994})
protein.subtract_metabolites({model.metabolites.get_by_id('pi_c'): 15.994})
protein.subtract_metabolites({model.metabolites.get_by_id('ppi_c'): 7.997})

protein.add_metabolites({model.metabolites.get_by_id('alatrna_c'): -0.516815632})
protein.add_metabolites({model.metabolites.get_by_id('argtrna_c'): -0.304153341})
protein.add_metabolites({model.metabolites.get_by_id('asntrna_c'): -0.176776316})
protein.add_metabolites({model.metabolites.get_by_id('asptrna_c'): -0.280336574})
protein.add_metabolites({model.metabolites.get_by_id('cystrna_c'): -0.031387457})
protein.add_metabolites({model.metabolites.get_by_id('glntrna_c'): -0.156194449})
protein.add_metabolites({model.metabolites.get_by_id('glutrna_c'): -0.303870745})
protein.add_metabolites({model.metabolites.get_by_id('glytrna_c'): -0.397806412})
protein.add_metabolites({model.metabolites.get_by_id('histrna_c'): -0.10032933})
protein.add_metabolites({model.metabolites.get_by_id('iletrna_c'): -0.284016986})
protein.add_metabolites({model.metabolites.get_by_id('leutrna_c'): -0.393629113})
protein.add_metabolites({model.metabolites.get_by_id('lystrna_c'): -0.310665435})
protein.add_metabolites({model.metabolites.get_by_id('mettrna_c'): -0.114331619})
protein.add_metabolites({model.metabolites.get_by_id('phetrna_c'): -0.152929045})
protein.add_metabolites({model.metabolites.get_by_id('protrna_c'): -0.219074766})
protein.add_metabolites({model.metabolites.get_by_id('sertrna_c'): -0.271575717})
protein.add_metabolites({model.metabolites.get_by_id('thrtrna_c'): -0.250518923})
protein.add_metabolites({model.metabolites.get_by_id('trptrna_c'): -0.041445843})
protein.add_metabolites({model.metabolites.get_by_id('tyrtrna_c'): -0.109668954})
protein.add_metabolites({model.metabolites.get_by_id('valtrna_c'): -0.370206544})
protein.add_metabolites({model.metabolites.get_by_id('gtp_c'): -9.57147})
protein.add_metabolites({model.metabolites.get_by_id('h2o_c'): -9.57147})

protein.add_metabolites({model.metabolites.get_by_id('trnaala_c'): 0.516815632})
protein.add_metabolites({model.metabolites.get_by_id('trnaarg_c'): 0.304153341})
protein.add_metabolites({model.metabolites.get_by_id('trnaasn_c'): 0.176776316})
protein.add_metabolites({model.metabolites.get_by_id('trnaasp_c'): 0.280336574})
protein.add_metabolites({model.metabolites.get_by_id('trnacys_c'): 0.031387457})
protein.add_metabolites({model.metabolites.get_by_id('trnagln_c'): 0.156194449})
protein.add_metabolites({model.metabolites.get_by_id('trnaglu_c'): 0.303870745})
protein.add_metabolites({model.metabolites.get_by_id('trnagly_c'): 0.397806412})
protein.add_metabolites({model.metabolites.get_by_id('trnahis_c'): 0.10032933})
protein.add_metabolites({model.metabolites.get_by_id('trnaile_c'): 0.284016986})
protein.add_metabolites({model.metabolites.get_by_id('trnaleu_c'): 0.393629113})
protein.add_metabolites({model.metabolites.get_by_id('trnalys_c'): 0.310665435})
protein.add_metabolites({model.metabolites.get_by_id('trnamet_c'): 0.114331619})
protein.add_metabolites({model.metabolites.get_by_id('trnaphe_c'): 0.152929045})
protein.add_metabolites({model.metabolites.get_by_id('trnapro_c'): 0.219074766})
protein.add_metabolites({model.metabolites.get_by_id('trnaser_c'): 0.271575717})
protein.add_metabolites({model.metabolites.get_by_id('trnathr_c'): 0.250518923})
protein.add_metabolites({model.metabolites.get_by_id('trnatrp_c'): 0.041445843})
protein.add_metabolites({model.metabolites.get_by_id('trnatyr_c'): 0.109668954})
protein.add_metabolites({model.metabolites.get_by_id('trnaval_c'): 0.370206544})
protein.add_metabolites({model.metabolites.get_by_id('gdp_c'): 9.57147})
protein.add_metabolites({model.metabolites.get_by_id('pi_c'): 9.57147})
protein.add_metabolites({model.metabolites.get_by_id('h_c'): 9.57147})
#print(protein.check_mass_balance())

In [ ]:
#Saving updated model with protein biomass precursor reaction changes and run on Parameterizing_GAM_nGAM.py
#cobra.io.write_sbml_model(model, 'i_ZM4_489_debug.xml')

#So far it is running and solvable for low glucose uptake values so not an issue with the updated protein synthesis precursor reaction
#Below slowly going to start changing biomass (begin with proteins and then will change lipids)

In [24]:
#Updating Lipids precursor synthesis reactions based off of JRV paper on lipid membrane remodeling in Z. mobilis
#More details refer to 'KF_biomass updates_GSM_RBA.xlsx'

#-----------------------
#Adding in PC as metabolite and a synthesis reactios
#made it orphan reaction (no GPR - gene not yet annotate in Zymo - also potentially other pathways to make pc but this one was simplest in KEGG)
#-----------------------
pc_ZM_c = Metabolite('pc_ZM_c',formula='None',name='Phosphatidylcholine (Z. mobilis)',compartment ='c', charge=0)

PCS = Reaction(
    'PCS',
    name = 'phosphatidylcholine synthase', 
    subsystem = 'Glycerophospholipid Metabolism', 
    lower_bound= 0, 
    upper_bound= 1000,
)  

PCS.notes['SUBSYSTEM'] = 'Glycerophospholipid Metabolism'

PCS.add_metabolites({model.metabolites.get_by_id('cdpdag1_c'): -1})
PCS.add_metabolites({pc_ZM_c: 1})

model.add_reactions([PCS])
#print(PCS)


#-----------------------
#Updating the Phospolipid synthesis reaction to reflect experimental data
#-----------------------
Phospholipids_ZM = model.reactions.get_by_id('Phospholipidsr_ZM')

Phospholipids_ZM.subtract_metabolites({model.metabolites.get_by_id('ps_ZM_c'): -0.092})
Phospholipids_ZM.add_metabolites({model.metabolites.get_by_id('pc_ZM_c'): -0.201})

Phospholipids_ZM.subtract_metabolites({model.metabolites.get_by_id('clpn_ZM_c'): -0.053})
Phospholipids_ZM.add_metabolites({model.metabolites.get_by_id('clpn_ZM_c'): -0.019})

Phospholipids_ZM.subtract_metabolites({model.metabolites.get_by_id('pe_ZM_c'): -0.777})
Phospholipids_ZM.add_metabolites({model.metabolites.get_by_id('pe_ZM_c'): -0.794})

Phospholipids_ZM.subtract_metabolites({model.metabolites.get_by_id('pg_ZM_c'): -0.377})
Phospholipids_ZM.add_metabolites({model.metabolites.get_by_id('pg_ZM_c'): -0.313})


#-----------------------
#Updating coefficients and metabolites for fatty acids based on percent comp. fatty acids found in experimental data (here the coefficients will represent percent compostion)
#-----------------------

AGPAT = model.reactions.get_by_id('AGPAT_ZM')
#print(AGPAT)
AGPAT.add_metabolites({model.metabolites.get_by_id('hdeACP_c'): -0.001})
AGPAT.subtract_metabolites({model.metabolites.get_by_id('myrsACP_c'): -0.05})
AGPAT.add_metabolites({model.metabolites.get_by_id('octeACP_c'): -0.016})
AGPAT.add_metabolites({model.metabolites.get_by_id('palmACP_c'): -0.014})
AGPAT.add_metabolites({model.metabolites.get_by_id('C190cACP_c'): -0.016})
AGPAT.add_metabolites({model.metabolites.get_by_id('tdeACP_c'): -0.003})
#print(AGPAT)

G3PAT = model.reactions.get_by_id('G3PAT_ZM')
#print(G3PAT)
G3PAT.add_metabolites({model.metabolites.get_by_id('hdeACP_c'): -0.001})
G3PAT.subtract_metabolites({model.metabolites.get_by_id('myrsACP_c'): -0.05})
G3PAT.add_metabolites({model.metabolites.get_by_id('octeACP_c'): -0.016})
G3PAT.add_metabolites({model.metabolites.get_by_id('palmACP_c'): -0.014})
G3PAT.add_metabolites({model.metabolites.get_by_id('C190cACP_c'): -0.016})
G3PAT.add_metabolites({model.metabolites.get_by_id('tdeACP_c'): -0.003})
#print(G3PAT)

#-----------------------
#For lipid biomass precursor reactions removing the TAGs reactions since Julio's paper showed no detectable TAGs (very small if are) under ZMM anaerobic
#Will aslo nedd to upate the Lipid_ZM_c synthesis reaction
#-----------------------
lipids = model.reactions.get_by_id('LIPIDr_ZM')
#print(lipids)
lipids.subtract_metabolites({model.metabolites.get_by_id('TAGs_ZM_c'): -0.0471})
lipids.add_metabolites({model.metabolites.get_by_id('Hopanoids_ZM_c'): -0.21})
lipids.subtract_metabolites({model.metabolites.get_by_id('Phospholipids_ZM_c'): -0.163})
#print(lipids)

#Going to keep TAGsr_ZM reaction in the model but for time being and based on experimental evidence set the flux to 0 
TAGsr = model.reactions.get_by_id('TAGsr_ZM')
TAGsr.bounds = (0,0)
#print(TAGsr.bounds)

#print(lipids)


In [ ]:
#Saving updated model with protein biomass and lipid precursor reaction changes and run on Parameterizing_GAM_nGAM.py
#cobra.io.write_sbml_model(model, 'i_ZM4_489_debug.xml')
#Note: need to run model with G6PDH2r and G6PDH2xr set to only run in the positive direction because if not get infinite loops with FNOR, RNF, ATPsynthase

#It is running!!!!

In [25]:
model

Name,iZM4_478
Memory address,186c759d0
Number of metabolites,806
Number of reactions,872
Number of genes,489
Number of groups,39
Objective expression,1.0*BIOMASS_ZM - 1.0*BIOMASS_ZM_reverse_34f17
Compartments,"Extracellular, Cytoplasm, Periplasm"


In [26]:
#Paramaterized the GAM and nGAM values so adding updated values of those into model
#For code see 'Parameterizing_GAM_nGAM.py'

ATPM = model.reactions.get_by_id('ATPM')
Biomass = model.reactions.get_by_id('BIOMASS_ZM')

ATPM.bounds = (3.6335, 3.6335)

print(Biomass)

Biomass.add_metabolites({model.metabolites.get_by_id('atp_c'): -10.628346})
Biomass.add_metabolites({model.metabolites.get_by_id('h2o_c'): -10.628346})
Biomass.add_metabolites({model.metabolites.get_by_id('adp_c'): 10.628346})
Biomass.add_metabolites({model.metabolites.get_by_id('h_c'): 10.628346})
Biomass.add_metabolites({model.metabolites.get_by_id('pi_c'): 10.628346})

print(Biomass)

BIOMASS_ZM: 0.0549 CARBO_ZM_c + 0.0297 DNA_ZM_c + 0.0769 LIPID_ZM_c + 0.0275 PEPTIDO_ZM_c + 0.527 PROTEIN_ZM_c + 0.242 RNA_ZM_c + 0.0428 SM_MOLECULE_ZM_c + 75.368454365963 atp_c + 75.368454365963 h2o_c --> 75.368454365963 adp_c + 75.368454365963 h_c + 75.368454365963 pi_c
BIOMASS_ZM: 0.0549 CARBO_ZM_c + 0.0297 DNA_ZM_c + 0.0769 LIPID_ZM_c + 0.0275 PEPTIDO_ZM_c + 0.527 PROTEIN_ZM_c + 0.242 RNA_ZM_c + 0.0428 SM_MOLECULE_ZM_c + 85.996800365963 atp_c + 85.996800365963 h2o_c --> 85.996800365963 adp_c + 85.996800365963 h_c + 85.996800365963 pi_c


In [27]:
#Updating model id and saving model as .xml

model.id = 'iZM4_489'

cobra.io.write_sbml_model(model, 'i_ZM4_489.xml')

from cobra.io import save_json_model
save_json_model(model, 'i_ZM4_489.json', sort=True, pretty=True)

In [28]:
#Writing all modeled genes to text file so know which proteins need to model in RBA
filename = 'modeled_genes.txt'


with open(filename, 'w') as f: 
    for i in model.genes:
        f.write(f'{i}\n')

In [ ]:

#For the RBA model will need to eventually model isoprene and isobutanol production which are heterologous pathways
#Will add the reactions/pathways and save the model as separate one for heterologous pathways


#---------------------------------------
#Adding isoprene biosynthetic pathway (single step)
#---------------------------------------

# adding the Dimethylallyl diphospahte <=> Isoprene + Diphosphate
# Import existing metabolites
dmpp_c = model.metabolites.get_by_id ('dmpp_c')
ppi_c = model.metabolites.get_by_id ('ppi_c')

# Create new metabolite for isoprene
isop_c = Metabolite ('isop_c',formula='C5H8', name='Isoprene',compartment='c', charge=0)

ISOP_R = Reaction(
    'ISOP_R',
    name = 'Isoprene synthesis', 
    subsystem = 'Heterologous Pathways', 
    lower_bound= 0, 
    upper_bound= 1000,
)  

ISOP_R.notes['SUBSYSTEM'] = 'Heterologous Pathways'
ISOP_R.gene_reaction_rule = 'IspS'

ISOP_R.add_metabolites ({dmpp_c:-1.0, isop_c: 1.0, ppi_c:1.0})
model.add_reactions([ISOP_R])
#print(ISOP_R.check_mass_balance())

#Adding in the reaction (exchange) for the release of isoprene as a gas
#Create new metabolites for isop_p and isop_e
isop_p = Metabolite ('isop_p',formula='C5H8', name='Isoprene',compartment='p')
isop_e = Metabolite('isop_e',formula='C5H8', name='Isoprenne',compartment='e',charge=0)

ISOP_tPP = Reaction(
    'ISOP_tPP',
    name = 'Isoprene transport via diffusion (periplasm)', 
    subsystem = 'Heterologous Pathways', 
    lower_bound= -1000, 
    upper_bound= 1000,
) 

ISOP_tPP.notes['SUBSYSTEM'] = 'Heterologous Pathways'

ISOP_tPP.add_metabolites ({isop_c: -1.0, isop_p:1.0})
model.add_reactions([ISOP_tPP])
#print(ISOP_tPP.check_mass_balance())

ISOP_ePP = Reaction(
    'ISOP_ePP',
    name = 'Isoprene transport via diffusion (extracellular to periplasm)', 
    subsystem = 'Heterologous Pathways', 
    lower_bound= -1000, 
    upper_bound= 1000,
) 

ISOP_ePP.notes['SUBSYSTEM'] = 'Heterologous Pathways'

ISOP_ePP.add_metabolites ({isop_p: -1.0, isop_e:1.0})
model.add_reactions([ISOP_ePP])
#print(ISOP_ePP.check_mass_balance())

EX_ISOP_e = Reaction(
    'EX_ISOP_e',
    name = 'Isoprene exchange', 
    subsystem = 'Exchange', 
    lower_bound= 0, 
    upper_bound= 1000,
) 

EX_ISOP_e.notes['SUBSYSTEM'] = 'Heterologous Pathways'

EX_ISOP_e.add_metabolites ({isop_e: -1.0})
model.add_reactions([EX_ISOP_e])
#print(ISOP_ePP.check_mass_balance())

print(model.reactions.get_by_id('ISOP_R'))
print(model.reactions.get_by_id('ISOP_tPP'))
print(model.reactions.get_by_id('ISOP_ePP'))
print(model.reactions.get_by_id('EX_ISOP_e'))

model.id = 'iZM4_489_isoprene'

cobra.io.write_sbml_model(model, 'i_ZM4_489_isoprene.xml')



In [29]:
#---------------------------------------
#Adding isobutanol biosynthetic pathway (all reactions with heterologous genes)
#---------------------------------------
# Import existing reactions that will modify
als = model.reactions.get_by_id('ACLS')
kara1 = model.reactions.get_by_id('KARA1')
# Import existing metabolites, 3-Methyl-2-oxobutanoate (KIV), DHIV
# Import existing metabolite, CO2
mob_c = model.metabolites.get_by_id ('3mob_c')
co2_c = model.metabolites.get_by_id ('co2_c')
h_c = model.metabolites.get_by_id('h_c')
dhiv = model.metabolites.get_by_id('23dhmb_c')

#Adding the ALS reactions catalyzed by the heterologous KpALS
als_heterologous = als.copy()
als_heterologous = Reaction(
    'ALS_heterologous',
    name = 'Kp_Als_IBA_pathway',
    subsystem='Heterologous Pathways',
    lower_bound=0,
    upper_bound=1000,
)
als_heterologous.notes['SUBSYSTEM'] = 'Heterologous Pathways'
als_heterologous.gene_reaction_rule = ('Kp_ALS')

co2_c = model.metabolites.get_by_id ('co2_c')
pyr_c = model.metabolites.get_by_id('pyr_c')
alac = model.metabolites.get_by_id('alac__S_c')

als_heterologous.add_metabolites({pyr_c:-2.0, h_c:-1.0, alac:1.0, co2_c:1.0 })
model.add_reactions([als_heterologous])
print(als_heterologous.check_mass_balance())
print(als_heterologous)


#Adding the NADH dependant IlvC reaction catalyzed by the heterologus EcoIlvC (engineered by Arnold lab be NADH dependant)
ilvC = kara1.copy()
ilvC = Reaction(
    'IlvC_heterologous',
    name = 'Eco_IlvC_IBA_pathway',
    subsystem = 'Heterologous Pathways',
    lower_bound=-1000,
    upper_bound=0,
)

ilvC.notes['SUBSYSTEM'] = 'Heterologous Pathways'
ilvC.gene_reaction_rule = ('Eco_IlvC')

#Import existing metabolites
nadh_c = model.metabolites.get_by_id ('nadh_c')
nad_c = model.metabolites.get_by_id ('nad_c')


ilvC.add_metabolites ({dhiv: -1.0, nad_c: -1.0, h_c:1.0, alac:1.0,nadh_c:1.0 })
print(ilvC)
print(ilvC.check_mass_balance())
model.add_reactions([ilvC])


# Add new metabolites, Isobutyraldehyde
ibdh_c = Metabolite( 'ibdh_c', formula='C4H8O', name='Isobutyraldehyde', compartment ='c',charge=0)

AKDC = Reaction(
    'AKDC',
    name = 'alpha-keto acid decarboxylase', 
    subsystem = 'Heterologous Pathways', 
    lower_bound= 0, 
    upper_bound= 1000,
)  

AKDC.notes['SUBSYSTEM'] = 'Heterologous Pathways'

AKDC.add_metabolites ({mob_c: -1.0, h_c: -1.0, ibdh_c: 1.0, co2_c: 1.0 })
AKDC.gene_reaction_rule = ('Lla_KivD')
model.add_reactions([AKDC])
#print(AKDC.check_mass_balance())


# Add New Metabolite Isobutanol
isob_c = Metabolite ('isob_c',formula='C4H10O',name='Isobutanol',compartment ='c')
# Import Existing Metabolites
ibdh_c = model.metabolites.get_by_id ('ibdh_c')


ACDH = Reaction(
    'ACHD',
    name = 'Alcohol dehydrogenase', 
    subsystem = 'Heterologous Pathways', 
    lower_bound= 0, 
    upper_bound= 1000,
)  

ACDH.notes['SUBSYSTEM'] = 'Heterologous Pathways'

ACDH.add_metabolites ({ibdh_c: -1.0, nadh_c: -1.0, h_c:-1.0, isob_c: 1.0, nad_c:1.0})
ACDH.gene_reaction_rule = ('Lla_AdhA')
model.add_reactions([ACDH])
#print(ACDH.check_mass_balance())


#Adding in the reaction (exchange) for the release of isobutanol as a gas
#Create new metabolites for isob_p and isob_e
isob_p = Metabolite ('isob_p',formula='C4H10O', name='Isobutanol',compartment='p')
isob_e = Metabolite('isob_e',formula='C4H10O', name='Isobutanol',compartment='e')

ISOB_tPP = Reaction(
    'ISOB_tPP',
    name = 'Isobutanol transport via diffusion (periplasm)', 
    subsystem = 'Heterologous Pathways', 
    lower_bound= -1000, 
    upper_bound= 1000,
) 

ISOB_tPP.notes['SUBSYSTEM'] = 'Heterologous Pathways'

ISOB_tPP.add_metabolites ({isob_c: -1.0, isob_p:1.0})
model.add_reactions([ISOB_tPP])
#print(ISOB_tPP.check_mass_balance())

ISOB_ePP = Reaction(
    'ISOB_ePP',
    name = 'Isobutanol transport via diffusion (extracellular to periplasm)', 
    subsystem = 'Heterologous Pathways', 
    lower_bound= -1000, 
    upper_bound= 1000,
) 

ISOB_ePP.notes['SUBSYSTEM'] = 'Heterologous Pathways'

ISOB_ePP.add_metabolites ({isob_p: -1.0, isob_e:1.0})
model.add_reactions([ISOB_ePP])
#print(ISOB_ePP.check_mass_balance())

EX_ISOB_e = Reaction(
    'EX_ISOB_e',
    name = 'Isobutanol exchange', 
    subsystem = 'Heterologous Pathways', 
    lower_bound= 0, 
    upper_bound= 1000,
) 

EX_ISOB_e.notes['SUBSYSTEM'] = 'Heterologous Pathways'

EX_ISOB_e.add_metabolites ({isob_e: -1.0})
model.add_reactions([EX_ISOB_e])
print(EX_ISOB_e)
#print(ISOB_ePP.check_mass_balance())

model.id = 'iZM4_489_isobutanol'

cobra.io.write_sbml_model(model, 'i_ZM4_489_isobutanol.xml')

{}
ALS_heterologous: h_c + 2.0 pyr_c --> alac__S_c + co2_c
IlvC_heterologous: 23dhmb_c + nad_c <-- alac__S_c + h_c + nadh_c
{}
EX_ISOB_e: isob_e --> 
